In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
import openpyxl
import warnings
warnings.filterwarnings('ignore')

### Before All Questions: Load Data from Excel

In [2]:
wb = openpyxl.load_workbook('CMSC 5718 Assignment 2 parameters.xlsx', data_only=True)

# Load stock prices
ws_prices = wb['stock prices']
stock_codes = [267, 388, 700, 941, 981, 1801, 2319, 2899, 9888, 9988]
stock_names = ['Citic Limited', 'Hong Kong Exchanges', 'Tencent', 'China Mobile',
               'SMIC', 'Innovent Biologics', 'China Mengniu Dairy', 'Zijin Mining',
               'Baidu', 'Alibaba']

# Parse price data (rows 4 onwards have data, row 2 has stock codes, row 3 has "Dates")
dates = []
price_data = {code: [] for code in stock_codes}
hsi_prices = []

for i, row in enumerate(ws_prices.iter_rows(values_only=True), 1):
    if i < 4:  # Skip header rows
        continue
    row_list = list(row)
    date_val = row_list[1]
    if date_val is None:
        continue
    if hasattr(date_val, 'strftime'):
        dates.append(date_val)
    else:
        from datetime import datetime
        try:
            dates.append(datetime.strptime(str(date_val), '%Y-%m-%d'))
        except:
            continue
    
    for j, code in enumerate(stock_codes):
        price_data[code].append(float(row_list[j + 2]))
    hsi_prices.append(float(row_list[12]))

# Create DataFrame
price_df = pd.DataFrame(price_data, index=dates)
price_df['HSI'] = hsi_prices

print(f"\nData loaded: {len(dates)} trading days from {dates[0].strftime('%Y-%m-%d')} to {dates[-1].strftime('%Y-%m-%d')}")
print(f"Number of stocks: {len(stock_codes)}")
print(f"Stock codes: {stock_codes}")

# Load expected returns
ws_er = wb['Consensus Expected return']
expected_returns = {}
for row in ws_er.iter_rows(values_only=True):
    row_list = list(row)
    if row_list[2] is not None and isinstance(row_list[2], (int, float)):
        code = int(row_list[2])
        if code in stock_codes:
            expected_returns[code] = float(row_list[4])

rf = 0.029  # Risk-free rate

print(f"\nExpected returns:")
for code in stock_codes:
    print(f"  {code} ({stock_names[stock_codes.index(code)]}): {expected_returns[code]*100:.2f}%")
print(f"  Risk-free rate: {rf*100:.2f}%")



Data loaded: 493 trading days from 2023-12-29 to 2025-12-31
Number of stocks: 10
Stock codes: [267, 388, 700, 941, 981, 1801, 2319, 2899, 9888, 9988]

Expected returns:
  267 (Citic Limited): 27.50%
  388 (Hong Kong Exchanges): 35.50%
  700 (Tencent): 31.00%
  941 (China Mobile): 11.50%
  981 (SMIC): 41.10%
  1801 (Innovent Biologics): 33.81%
  2319 (China Mengniu Dairy): 29.58%
  2899 (Zijin Mining): 21.30%
  9888 (Baidu): 37.50%
  9988 (Alibaba): 46.90%
  Risk-free rate: 2.90%
